# Capstone — mirrors your deployed research paper

This notebook mirrors the deployed research paper (ML-11, `docs/index.html`) end to end:
question → data → methodology → results vs the baseline → limitations → recommendations →
the artifacts the paper embeds.

Every number that can be recomputed from the saved analysis outputs **is recomputed here**
(base rates, the baseline rule, flags, precision@K, reason codes, all five figures).
Numbers that require the gated warehouse (the held-out model AUCs and precision@20) are
**restated from the w05/w06 notebook runs with their source named**.

Simple words, honest numbers. No client-identifying data appears anywhere in this file.


In [1]:
from pathlib import Path
import pandas as pd


def find_outputs() -> Path:
    """Locate work/outputs by walking up from the current working directory."""
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        p = cand / "work" / "outputs"
        if p.is_dir():
            return p
    raise FileNotFoundError(
        "work/outputs not found. Regenerate it first by running the warehouse query "
        "notebooks (skills/querying-big-datasets) against the gated release."
    )


OUT = find_outputs()
ROOT = OUT.parent.parent
DOCS_CHARTS = ROOT / "docs" / "charts"

base_df = pd.read_csv(OUT / "baseline_action_score.csv")   # 108,254 eligible pages, score + label
queue_df = pd.read_csv(OUT / "action_playbook_queue.csv")  # the ranked action queue

print("analysis outputs :", OUT.relative_to(ROOT))
print("eligible pages   :", f"{len(base_df):,}")
print("base rate        :", f"{base_df.declined_30d.mean():.3f}")


analysis outputs : work\outputs
eligible pages   : 108,254
base rate        : 0.674


## 1. Question

*The research question and the decision it supports.*

**Question:** which page should a content editor *look at first* when the goal is to keep
high-traffic content from losing search impressions?

- **Decision supported:** which page to review for a refresh **this week**.
- **Unit of analysis:** one page at one decision date `t = 2026-05-31`.
- **Output:** a ranked, reason-coded refresh queue. It only decides *order*; the human makes the edit.
- **Cost of a wrong call:** asymmetric. Choosing a page that would not have moved costs editor-hours
  (recoverable). Skipping a page that keeps losing traffic answers to nothing until the loss is
  already visible in next month's report.
- **Why data/ML?** The queue is built from observed search signals, and we can *measure* how often
  "ranked first" really meant "declined next window" — on a held-out set of clients.


In [2]:
print("research question: which pages should a content editor look at first?")
print("  unit of analysis : one page at one decision date t = 2026-05-31")
print("  output           : ranked, reason-coded refresh queue (ordering only)")
print("  wrong-call cost  : asymmetric - hours are recoverable;")
print("                     a visible page skipped keeps losing traffic until next month's report")
print(f"  base rate        : {base_df.declined_30d.mean():.3f} (the bar every metric must clear)")


research question: which pages should a content editor look at first?
  unit of analysis : one page at one decision date t = 2026-05-31
  output           : ranked, reason-coded refresh queue (ordering only)
  wrong-call cost  : asymmetric - hours are recoverable;
                     a visible page skipped keeps losing traffic until next month's report
  base rate        : 0.674 (the bar every metric must clear)


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** full FlyRank ML Internship warehouse, build **v20260703**, gated
(`FlyRank/internship-warehouse`), queried remotely via DuckDB.

| Table | Rows | Role |
|---|---|---|
| `fact_content_daily_performance` | 78,835,655 | daily per-page search metrics — window features + label |
| `dim_content` | 519,606 | static content descriptors |
| `dim_clients` | 104 | grouping key for the split — never a feature |

**Reduced to one decision snapshot:** 108,254 eligible pages with `impressions ≥ 100` in B and
`≥ 15` Google Search Console days in B.

**Windows:** B = (2026-05-01, 2026-05-31] features · `t = 2026-05-31` · F = (2026-05-31, 2026-06-30] label.

**Excluded on purpose:** `fact_content_query_90d` (its fixed window contains F at this `t`),
`imp_f` and all F-derived aggregates (they *are* the label), the `ga4_*`/`sessions_*`/`ai_*`/
`scroll_events` family (zero-filled outside GA4 availability, ~74% off), `provider_used`/
`model_used` (circular + privacy risk), workflow timestamps, both hash-ID columns as features
(grouping only), and anything non-anonymized.

**Public-safe:** the column check below confirms no URLs, titles, domains, keywords, or queries
appear anywhere in the analysis outputs.


In [3]:
print("files used:")
print("  baseline_action_score.csv ->", base_df.shape)
print("  action_playbook_queue.csv ->", queue_df.shape)
print("columns:", list(base_df.columns))

banned = ["url", "title", "domain", "query", "keyword", "client_name", "brand"]
hits = [c for c in base_df.columns if any(b in c.lower() for b in banned)]
print("public-safety check (banned column names present):", hits or "none - clean")

print("\nwhy pages enter the queue (reason codes, full eligible population):")
print(base_df.reason.fillna("(none)").value_counts().head(5).to_string())

defects = [
    "June ships duplicated: 6,390 identical groups (0.05% of June rows) - deduped before aggregation",
    "GA4 engagement columns are zero-filled, not absent - entire ga4_* family excluded",
    "Unbalanced panel: 37/104 clients lack a GSC anchor date - hence per-page eligibility, not a global calendar",
    "Freshness timestamps are release-build state - used directionally only",
]
print("\ndefects found and handled (each measured, not assumed):")
for d in defects:
    print("  -", d)


files used:
  baseline_action_score.csv -> (108254, 8)
  action_playbook_queue.csv -> (108254, 9)
columns: ['client_hash_id', 'content_hash_id', 'imp_b', 'age_days', 'pos_avg_b', 'score', 'reason', 'declined_30d']
public-safety check (banned column names present): none - clean

why pages enter the queue (reason codes, full eligible population):
reason
stale+position_slipping                22464
position_slipping                      18196
has_traffic                            15629
has_traffic+stale                      14474
has_traffic+stale+position_slipping    12930

defects found and handled (each measured, not assumed):
  - June ships duplicated: 6,390 identical groups (0.05% of June rows) - deduped before aggregation
  - GA4 engagement columns are zero-filled, not absent - entire ga4_* family excluded
  - Unbalanced panel: 37/104 clients lack a GSC anchor date - hence per-page eligibility, not a global calendar
  - Freshness timestamps are release-build state - used directiona

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions.** One snapshot (the queue ranks *now*, re-run each month). A feature is legal iff it
is knowable at or before `t`. Association, not causation — no intervention variable exists in the data.

**Label, one sentence.** `declined_30d = 1` iff `imp_f < 0.8 × imp_b` — impressions fall more than
20% in the 30 days after `t` versus the baseline month. Base rate: **67.4%**.

**Features (22, all B-window or static).** traffic/position in B (`imp_b, clk_b, ctr_b, gsc_days_b,
pos_avg_b, pos_vol_b`); static content descriptors with `has_*` companion flags for the ~22–31% that
are missing (`word_count … backlinks`); age/freshness (`age_days, days_since_update`); three
label-encoded categoricals (`content_type, main_intent, competition_level`).

**Baseline — a rule a person can read (frozen before any model).**

    flag  = has_traffic(imp_b ≥ 600) AND stale(age_days ≥ 180) AND position_slipping(pos_avg_b ≥ 12)
    score = flag × imp_b                      # volume decides order within the flag

**Validation.** GroupKFold(5) grouped **by client** — a random split overstates skill by ~0.09 AUC
because rows from one client share hidden character. The grouped split measures the production case:
*clients the model never trained on*. Features from B (≤ t), label from F (> t) — time-aware by construction.
The baseline's frozen score is recomputed on the same test rows each fold.

**Leakage probes** (the confession test, logistic regression, same legal set) — measured in
`w06_validation_audit.ipynb` and restated below.


In [4]:
print("label definition: declined_30d = imp_f < 0.8 * imp_b  (forward window F is never read as a feature)")
print(f"  base rate: {base_df.declined_30d.mean():.4f}")

# Recompute the frozen baseline rule from the raw columns and check it equals the saved score.
has_traffic = base_df.imp_b >= 600
stale = base_df.age_days >= 180
slipping = base_df.pos_avg_b >= 12
flag = has_traffic & stale & slipping
recomputed_score = flag.astype(int) * base_df.imp_b
print("\nrule: flag = traffic(>=600) AND stale(>=180d) AND slipping(pos_avg>=12); score = flag * imp_b")
print(f"  recomputed flags: {(recomputed_score > 0).sum():,} | saved flags: {(base_df.score > 0).sum():,} | match: {(recomputed_score > 0).eq(base_df.score > 0).all()}")

print("\nfeature groups (22 total, all knowable at t):")
for g in [
    "  traffic/position (B): imp_b, clk_b, ctr_b, gsc_days_b, pos_avg_b, pos_vol_b",
    "  content descriptors (static, + has_* flags for ~22-31% missingness): word_count, char_count,",
    "      keyword_char_count, keyword_token_count, url_char_count, category_count, search_volume, backlinks",
    "  age/freshness: age_days, days_since_update      categorical: content_type, main_intent, competition_level",
]:
    print(g)
print("  left out on purpose: imp_f + all F-aggregates, query table, ga4 family, product flags, both hash IDs")

probes = pd.DataFrame([
    ("legal set, random split",          0.632, 0.686, "barely above chance - honest and unimpressive"),
    ("legal set, grouped by client",     0.539, 0.676, "the ~0.09 AUC drop is the memorization finding"),
    ("+ imp_f (the label's own input)",  0.998, 0.924, "~1.0 - the classic leakage confession"),
    ("+ query-table-style forward signal", 0.873, 0.865, "any forward signal inflates"),
    ("+ provider_used / model_used",     0.652, 0.692, "circular + privacy risk - excluded"),
], columns=["configuration", "ROC AUC", "avg precision", "reading"])
print("\nleakage probes (logistic regression, same legal set; measured in w06 notebook):")
print(probes.to_string(index=False))


label definition: declined_30d = imp_f < 0.8 * imp_b  (forward window F is never read as a feature)
  base rate: 0.6744

rule: flag = traffic(>=600) AND stale(>=180d) AND slipping(pos_avg>=12); score = flag * imp_b
  recomputed flags: 12,930 | saved flags: 12,930 | match: True

feature groups (22 total, all knowable at t):
  traffic/position (B): imp_b, clk_b, ctr_b, gsc_days_b, pos_avg_b, pos_vol_b
  content descriptors (static, + has_* flags for ~22-31% missingness): word_count, char_count,
      keyword_char_count, keyword_token_count, url_char_count, category_count, search_volume, backlinks
  age/freshness: age_days, days_since_update      categorical: content_type, main_intent, competition_level
  left out on purpose: imp_f + all F-aggregates, query table, ga4 family, product flags, both hash IDs

leakage probes (logistic regression, same legal set; measured in w06 notebook):
                     configuration  ROC AUC  avg precision                                        reading


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Everyone is judged on the **same five client-grouped folds**, with the **base rate (0.67)** next to
every number. Headline metric: precision@20 (the top of the list the editor actually uses) and
ROC AUC (ranking depth).

**Held-out** (measured in `w05_model.ipynb` / `w06_validation_audit.ipynb`):

| Delivered by | Precision@20 | Precision@50 | ROC AUC |
|---|---|---|---|
| base rate (no model) | 0.67 | 0.67 | 0.50 (chance) |
| baseline rule (transparent) | **0.73** | **0.73** | — |
| logistic regression | 0.65 | — | 0.54 |
| random forest (300 trees) | 0.70 | — | 0.61 |

The learned model does **not** clear the transparent baseline at top-K; the forest's only clear edge
is ranking depth. The in-sample playbook (recomputed below from the saved queue) shows the rule holds
a ~1.15–1.26× lift over the base rate within its own population — numbers never quoted without the
held-out counterparts above them.


In [5]:
heldout = pd.DataFrame([
    ("base rate (no model)",          0.67, 0.67, "0.50 (chance)"),
    ("baseline rule (transparent)",   "0.73", "0.73", "-"),
    ("logistic regression",           0.65, "-", 0.54),
    ("random forest (300 trees)",     0.70, "-", 0.61),
], columns=["delivered by", "precision@20", "precision@50", "ROC AUC"])
print("held-out, mean over 5 client-grouped folds (source: w05/w06 notebook runs):")
print(heldout.to_string(index=False))

order = queue_df.score.argsort()[::-1]
lbl = queue_df.declined_30d.iloc[order].reset_index(drop=True)
base = float(lbl.mean())
rows = []
for k in [10, 20, 50, 100, 200, 500]:
    p = float(lbl.head(k).mean())
    rows.append((k, round(p, 3), round(p / base, 2)))
pk = pd.DataFrame(rows, columns=["K", "precision", "lift vs base rate"])
print("\nin-sample ranked-queue precision@K (recomputed from the saved queue):")
print(pk.to_string(index=False))


held-out, mean over 5 client-grouped folds (source: w05/w06 notebook runs):
               delivered by precision@20 precision@50       ROC AUC
       base rate (no model)         0.67         0.67 0.50 (chance)
baseline rule (transparent)         0.73         0.73             -
        logistic regression         0.65            -          0.54
  random forest (300 trees)          0.7            -          0.61

in-sample ranked-queue precision@K (recomputed from the saved queue):
  K  precision  lift vs base rate
 10      0.800               1.19
 20      0.850               1.26
 50      0.780               1.16
100      0.810               1.20
200      0.775               1.15
500      0.800               1.19


## 5. Limitations

*What this work cannot claim.*

- **Ranks, not causes.** No experiment exists in the data. The honest form is "these pages look
  worth an editor reviewing first" — nothing here shows that editing restores rankings.
- **One snapshot** at `t`. It says nothing about next quarter without a re-run.
- **New-client generalization is the weakest case** (grouped AUC ~0.61 vs the 0.79 random-split mirage).
- **~88% of eligible pages fall outside the rule** (12,930 of 108,254). Unflagged means *off the
  rule*, not *no opportunity*.
- **Misses exist.** ~15% of the top-20 held steady — mostly huge, old, off-page-1 pages.
- **Thresholds (600 / 180 / 12) are settings** read off this data period, not laws.
- **Censoring inflates the label.** Pages deleted inside F count as declines; the effect is ~1 page
  here but real in general.
- **No per-client calibration, no real-time retraining** — a frozen, auditable artifact.


In [6]:
flagged = int((base_df.score > 0).sum())
print(f"flagged by the rule : {flagged:,}  ({flagged / len(base_df):.1%} of eligible)")
print(f"outside the rule    : {len(base_df) - flagged:,}  ({(1 - flagged / len(base_df)):.1%}) - off the rule, not 'no opportunity'")

order = queue_df.score.argsort()[::-1]
lbl = queue_df.declined_30d.iloc[order].reset_index(drop=True)
p20 = float(lbl.head(20).mean())
misses = 20 - int(round(p20 * 20))
print(f"top-20 in-sample precision@20: {p20:.2f}  ->  {misses}/20 top picks held steady in-sample ('misses')")
print("censoring: pages deleted inside F count as declines; exactly 1 of 108,254 was deleted in this snapshot")


flagged by the rule : 12,930  (11.9% of eligible)
outside the rule    : 95,324  (88.1%) - off the rule, not 'no opportunity'
top-20 in-sample precision@20: 0.85  ->  3/20 top picks held steady in-sample ('misses')
censoring: pages deleted inside F count as declines; exactly 1 of 108,254 was deleted in this snapshot


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

1. **Run the transparent rule as triage** — and verify each pick by hand (live & indexable? already
   refreshed? position reading real? does the demand still exist? are we allowed to touch it?).
2. **Use the forest for the middle of the queue, not the top** (its edge is ranking depth, AUC 0.61 vs 0.54).
3. **Watch the blind spots** — any flagged page at ≥60k impressions should be checked personally.
4. **Never automate destructive actions** from this score — `declined` is not "delete this page".
5. **Re-run monthly** against a fresh warehouse build; numbers here are tied to one build.
6. **Treat 600/180/12 as settings** to tune on the portfolio, not a recipe.
7. **Close the causal gap** with a randomized or matched evaluation of refresh edits.


In [7]:
acts = queue_df[queue_df.score > 0].sort_values("score", ascending=False)
print("top of the ranked queue (fields behind each recommendation; no client/content IDs shown):")
print(acts[["imp_b", "age_days", "pos_avg_b", "score", "reason", "action"]]
      .head(8).to_string(index=False))

reason_note = {"has_traffic": "real traffic", "stale": "old (stale)", "position_slipping": "off page-1"}
def human(code):
    return ", ".join(reason_note[c] for c in code.split("+"))

print("\nreason-code map (machine-readable twin of the plain-language recommendation):")
for code, n in base_df.reason.value_counts().head(5).items():
    print(f"  {code:<42} {int(n):>6,}  ->  refresh first: {human(code)}")


top of the ranked queue (fields behind each recommendation; no client/content IDs shown):
   imp_b  age_days  pos_avg_b    score                              reason                                                        action
151672.0       290  23.888427 151672.0 has_traffic+stale+position_slipping refresh first | earns real traffic | old (stale) | off page-1
121749.0       202  14.357788 121749.0 has_traffic+stale+position_slipping refresh first | earns real traffic | old (stale) | off page-1
 98599.0       207  16.640345  98599.0 has_traffic+stale+position_slipping refresh first | earns real traffic | old (stale) | off page-1
 97393.0       220  26.064530  97393.0 has_traffic+stale+position_slipping refresh first | earns real traffic | old (stale) | off page-1
 96732.0       298  16.178146  96732.0 has_traffic+stale+position_slipping refresh first | earns real traffic | old (stale) | off page-1
 86533.0       214  22.498653  86533.0 has_traffic+stale+position_slipping refresh first

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed page embeds five figures. This section regenerates **all five** into `docs/charts/`
in the current purple editorial palette:

| File | Numbers |
|---|---|
| `validation_choice_auc.png` | probe design values from w06 (restated) |
| `model_vs_baseline_precision.png` | held-out means from w05/w06 (restated) |
| `model_vs_baseline_auc.png` | held-out means from w05/w06 (restated) |
| `playbook_precision_at_k.png` | **recomputed here** from the saved queue |
| `playbook_reason_codes.png` | **recomputed here** from `baseline_action_score.csv` |


In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

INK   = "#2a2240"; MUTED  = "#6b6277"; GRID  = "#e9e1f3"
P_BASE = "#a9a0b6"; P_RULE = "#6b3ea8"; P_LR  = "#b8912f"; P_RF  = "#431e6e"

plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "figure.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "text.color": INK, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED})

def style_ax(ax, ylabel):
    ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(axis="y", color=GRID, lw=0.7)
    ax.set_axisbelow(True)

def bars(ax, labels, vals, colors, ylim, val_fmt="{:.3f}", chance=None):
    style_ax(ax, ax.get_ylabel())
    b = ax.bar(labels, vals, color=colors, width=0.56)
    for bar, v in zip(b, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + ylim * 0.02, val_fmt.format(v),
                ha="center", va="bottom", fontweight="bold", color=INK)
    if chance is not None:
        ax.axhline(chance, ls="--", color="#5a5560", lw=1)
        ax.text(len(labels) - 0.28, chance + ylim * 0.03, "chance", ha="right", fontsize=10, color=MUTED)
    ax.set_ylim(0, ylim)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)

# Figure 1 - validation design probe (values measured in w06 notebook)
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.set_title("The same model, three validation designs", loc="left", weight="bold", color=INK)
bars(ax, ["Naive random split", "Grouped by client", "Leaky: forward feature"],
     [0.790, 0.615, 0.991], [P_BASE, P_RULE, P_RF], 1.12, chance=0.5)
plt.tight_layout()
fig.savefig(DOCS_CHARTS / "validation_choice_auc.png", bbox_inches="tight"); plt.close(fig)

# Figure 2 - held-out precision@20 (values from w05/w06)
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.set_title("Precision@20 on the same client-grouped split", loc="left", weight="bold", color=INK)
bars(ax, ["Base rate", "Baseline rule", "Logistic regression", "Random forest"],
     [0.672, 0.730, 0.650, 0.700], [P_BASE, P_RULE, P_LR, P_RF], 0.9)
plt.tight_layout()
fig.savefig(DOCS_CHARTS / "model_vs_baseline_precision.png", bbox_inches="tight"); plt.close(fig)

# Figure 3 - held-out AUC (values from w05/w06)
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.set_title("Ranking depth: ROC AUC on the same split", loc="left", weight="bold", color=INK)
bars(ax, ["Logistic regression", "Random forest"], [0.538, 0.614], [P_LR, P_RF], 0.7, chance=0.5)
plt.tight_layout()
fig.savefig(DOCS_CHARTS / "model_vs_baseline_auc.png", bbox_inches="tight"); plt.close(fig)

# Figure 4 - precision@K recomputed here
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.set_title("Ranked-queue precision@K vs base rate", loc="left", weight="bold", color=INK)
style_ax(ax, "precision")
ax.plot(pk.K, pk.precision, marker="o", color=P_RULE, lw=2, label="precision@K (rule)")
ax.axhline(base, color=MUTED, ls="--", lw=1.2, label=f"base rate {base:.2f}")
ax.legend(frameon=False, loc="center right")
ax.set_xscale("log"); ax.set_xticks(list(pk.K)); ax.set_xticklabels(list(pk.K), fontsize=9); ax.minorticks_off(); ax.set_ylim(0, 1.0)
plt.tight_layout()
fig.savefig(DOCS_CHARTS / "playbook_precision_at_k.png", bbox_inches="tight"); plt.close(fig)

# Figure 5 - reason-code distribution, recomputed here
reason_full = base_df["reason"]
rc_top = reason_full.value_counts().head(5).sort_values()
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.set_title("Why pages enter the ranked queue (reason codes)", loc="left", weight="bold", color=INK)
style_ax(ax, "pages")
b = ax.barh(rc_top.index, rc_top.values, color=P_RULE)
for bar, v in zip(b, rc_top.values):
    ax.text(v + max(rc_top.values) * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{int(v):,}", va="center", fontweight="bold", color=INK)
ax.set_xlim(0, max(rc_top.values) * 1.12)
plt.tight_layout()
fig.savefig(DOCS_CHARTS / "playbook_reason_codes.png", bbox_inches="tight"); plt.close(fig)

print("regenerated 5 charts into:", DOCS_CHARTS)
for p in sorted(DOCS_CHARTS.glob("*.png")):
    print("  ", p.name, round(p.stat().st_size / 1024, 1), "KB")


regenerated 5 charts into: C:\Users\Bogdan\Downloads\flyyy rank\Assignment-1\docs\charts
   model_vs_baseline_auc.png 39.3 KB
   model_vs_baseline_precision.png 45.5 KB
   playbook_precision_at_k.png 44.2 KB
   playbook_reason_codes.png 57.2 KB
   validation_choice_auc.png 43.2 KB


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
